The goal of this code is to reconstruct images from the masked sinograms. The runtime version of Python must be under Python 12 for TIGRE to work.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [ ]:
!git clone https://github.com/CERN/TIGRE.git
%cd TIGRE
!pip install .

In [ ]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import os
import tigre
import tigre.algorithms as alg
from ipywidgets import interact, IntSlider

In [ ]:
# load the masked sinograms
def load_masked_sinograms(base_filename, image_n):
  bin1_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin1_projection.npy")
  bin2_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin2_projection.npy")
  bin3_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin3_projection.npy")
  bin4_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin4_projection.npy")
  bin5_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin5_projection.npy")
  bin6_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin6_projection.npy")
  binall_image = np.load(base_filename+f"Masked_Sinograms/{chr(image_n+65)}_bin_all_projection.npy")

  return [bin1_image, bin2_image, bin3_image, bin4_image, bin5_image, bin6_image, binall_image]

In [ ]:
# the information about the images we want to reconstruct
FILENAME_BASE = '/content/gdrive/MyDrive/XCITE/Honours Project/Reconstruction/2_NEW_DETECTOR_RECON/2_DATA/S1728_ChickenDrumstick4_25_33_41_50_59_68_120kVp/'
SCAN_NUM = 4 # the scan session number
FIND_OFFSET = False
ANGLES = np.linspace(0, 2 * np.pi, 360)
NUM_IMAGES = 6

In [ ]:
# create geometry object
geo = tigre.geometry(high_resolution=True)

# set geometry parameters
geo.DSD = 1000  # distance Source Detector (mm)
geo.DSO = 1000-186 # distance Source Origin (mm)

# detector parameters
geo.nDetector = np.array([72, 192])  # number of pixels (px) # check the order, values are correct [72, 192]
geo.dDetector = np.array([0.33, 0.33])  # size of each pixel (mm)
geo.sDetector = geo.nDetector * geo.dDetector  # total size of the detector (mm)

# image parameters
geo.nVoxel = np.array([72, 512, 512])  # number of voxels (vx)
geo.sVoxel = np.array([((72*0.33)/geo.DSD)*geo.DSO, ((192*0.33)/geo.DSD)*geo.DSO, ((192*0.33)/geo.DSD)*geo.DSO])  # total size of the image (mm) # height, width, width (using equivalent triangles)
geo.dVoxel = geo.sVoxel / geo.nVoxel  # size of each voxel (mm)

# offsets
geo.offOrigin = np.array([0, 0, 0])  # offset of image from origin (mm)
geo.offDetector = np.array([0, 0])  # offset of Detector (mm), first is left/right, up/down

In [ ]:
# checking the offsets for image reconstruction
if FIND_OFFSET:
  offsets = np.arange(-1,1,0.05)
  offset_recon = []

  bin1_projection, bin2_projection, bin3_projection, bin4_projection, bin5_projection, bin6_projection,binall_image = load_masked_sinograms(FILENAME_BASE, NUM_IMAGES-1)

  # testing different offsets
  for offset in offsets:
    geo.offDetector = np.array([0, offset])

    data = np.transpose(bin4_projection, (0, 2, 1)) # trying checking the nosie when removing the projections
    reconstruction = alg.fdk(data, geo, ANGLES)
    offset_recon.append(reconstruction[36, :, :])

  offset_recon_stacked = np.stack(offset_recon, axis=0)
  interact(plot_slice, slice_index=IntSlider(min=0, max=offset_recon_stacked.shape[0]-1, step=1, description='X-Slice'), axis='x');



---
The code below reconstructs the image given the best detector offset. The best detector offset should be the same for all the images acquired in the same scan session. It also reconstructs the images while removing projections.


In [ ]:
import tigre.algorithms as alg
BEST_OFFSET = 0.05
geo.offDetector = np.array([0, BEST_OFFSET]) # correcting the geometry
N_PROJECTIONS = 8 # factor to redeuce the number of projections by
NUM_IMAGES = 6 # total number of images, including the air scans
AFTER_AIRSCAN = 0 # the index after the air scans

# making files to save the output
os.makedirs(f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/{360//N_PROJECTIONS}_Projections_Images_512x512", exist_ok=False)
os.makedirs(f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/{360//N_PROJECTIONS}_Projections_Sinograms", exist_ok=False)

for i in range(AFTER_AIRSCAN,NUM_IMAGES):
  bin1_projection, bin2_projection, bin3_projection, bin4_projection, bin5_projection, bin6_projection, binall_image = load_masked_sinograms(FILENAME_BASE, i)
  projections = [bin1_projection, bin2_projection, bin3_projection, bin4_projection, bin5_projection, bin6_projection, binall_image]

  recon_results = []
  reduced_sinograms = []

  # reconstructing the image
  for j in range(len(projections)):
    projection = projections[j]
    data = np.transpose(projection, (0, 2, 1))
    reconstruction = alg.fdk(data[::N_PROJECTIONS,:,:], geo, ANGLES[::N_PROJECTIONS])
    recon_results.append(reconstruction)
    reduced_sinograms.append(data[::N_PROJECTIONS,:,:])

  np.save(f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/{360//N_PROJECTIONS}_Projections_Images_512x512/{chr(i+65)}{SCAN_NUM}", recon_results)
  np.save(f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/{360//N_PROJECTIONS}_Projections_Sinograms/{chr(i+65)}{SCAN_NUM}", reduced_sinograms)
